🐦 Conteo de Aves Marinas — Flujo Completo: Preliminar → Roboflow → Validado

FASE 1: Predicciones preliminares con modelos especializados por clase

FASE 2: Subida de fotos + predicciones a Roboflow como pre-anotaciones

FASE 3: Descarga de anotaciones corregidas y conteo validado

Instrucciones:
1. Monta tu Drive
2. Configura la celda 1 (rutas, clases, modelos, confianzas, SAHI)
3. Ejecuta las celdas en orden

In [1]:
# ============================================================
# 0. INSTALACIÓN (ejecutar una sola vez por sesión)
# ============================================================
!pip install ultralytics sahi openpyxl roboflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.8 MB/s eta 0:00:00


In [20]:
# ============================================================
# 🔧 CELDA 1: CONFIGURACIÓN
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np

# --- Rutas base ---
RUTA_FOTOS    = Path('/content/drive/MyDrive/conteo de aves marinas/fotos')
RUTA_MODELOS  = Path('/content/drive/MyDrive/conteo de aves marinas/modelos')
RUTA_SALIDA   = Path('/content/drive/MyDrive/conteo de aves marinas/conteos_preliminares.xlsx')
RUTA_VALIDADO = Path('/content/drive/MyDrive/conteo de aves marinas/Conteos_validados.xlsx')

# --- 16 Clases (orden fijo) ---
CLASES = [
    'chuita', 'chuita adulta', 'cushuri adulto', 'cushuri juvenil',
    'gallinazo cabeza roja', 'gaviota peruana adulta', 'guanay adulto',
    'pelicano adulto', 'pelicano juvenil', 'pichon pinguino', 'pichon piquero',
    'pinguino adulto', 'pinguino juvenil', 'piquero adulto', 'piquero juvenil',
    'zarcillo'
]

# Mapeo clase -> índice (para archivos YOLO)
CLASE_A_IDX = {c: i for i, c in enumerate(CLASES)}

# --- CONFIGURACIÓN POR CLASE ---
# Especifica: modelo.pt, confianza, si usa SAHI, y parámetros SAHI
# Puedes asignar el mismo modelo a varias clases si lo deseas.
CONFIG = {
    'chuita':               {'modelo': '26x.pt', 'conf': 0.48, 'sahi': False},
    'chuita adulta':        {'modelo': '26x.pt', 'conf': 0.48, 'sahi': False},
    'cushuri adulto':       {'modelo': '26x.pt', 'conf': 0.48, 'sahi': False},
    'cushuri juvenil':      {'modelo': '26s.pt', 'conf': 0.48, 'sahi': False},
    'gallinazo cabeza roja':{'modelo': '26m.pt', 'conf': 0.48, 'sahi': False},
    'gaviota peruana adulta':{'modelo': '26m.pt', 'conf': 0.48, 'sahi': False},
    'guanay adulto':        {'modelo': '26s.pt', 'conf': 0.48, 'sahi': False},
    'pelicano adulto':      {'modelo': 'pelicano_adulto_x.pt', 'conf': 0.21, 'sahi': True},
    'pelicano juvenil':     {'modelo': '26s.pt', 'conf': 0.48, 'sahi': False},
    'pichon pinguino':      {'modelo': '26s.pt', 'conf': 0.48, 'sahi': False},
    'pichon piquero':       {'modelo': '26x.pt', 'conf': 0.48, 'sahi': False},
    'pinguino adulto':      {'modelo': '26x.pt', 'conf': 0.48, 'sahi': False},
    'pinguino juvenil':     {'modelo': '26x.pt', 'conf': 0.48, 'sahi': False},
    'piquero adulto':       {'modelo': 'piquero_adulto_l.pt', 'conf': 0.42, 'sahi': True},
    'piquero juvenil':      {'modelo': '26x.pt', 'conf': 0.48, 'sahi': False},
    'zarcillo': {
        'modelo': '26x.pt',
        'conf': 0.48,
        'sahi': True,
        'sahi_slice_h': 1280,
        'sahi_slice_w': 1280,
        'sahi_overlap': 0.2,
    },
}

# --- Parámetros SAHI por defecto (si no se especifican en la clase) ---
SAHI_DEFAULT = {
    'slice_h': 1280,
    'slice_w': 1280,
    'overlap': 0.2,
}

# --- Roboflow ---
ROBOFLOW_API_KEY      = 'aqS5A9hwE8ZophGkEIYO'      # ← Reemplaza
ROBOFLOW_WORKSPACE    = 'jhon-goicochea'     # ← Reemplaza
ROBOFLOW_PROJECT      = 'conteo-av-1'      # ← Reemplaza
ROBOFLOW_BATCH_NAME   = 'conteo_preliminar'

# Verificar rutas
print('Fotos  :', RUTA_FOTOS,   '✅' if RUTA_FOTOS.exists() else '❌')
print('Modelos:', RUTA_MODELOS, '✅' if RUTA_MODELOS.exists() else '❌')
print(f'Clases configuradas: {len(CONFIG)}')
for c, cfg in CONFIG.items():
    ruta = RUTA_MODELOS / cfg['modelo']
    print(f'  {c:25s} → {cfg["modelo"]:30s} conf={cfg["conf"]} sahi={cfg["sahi"]} {"✅" if ruta.exists() else "❌"}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fotos  : /content/drive/MyDrive/conteo de aves marinas/fotos ✅
Modelos: /content/drive/MyDrive/conteo de aves marinas/modelos ✅
Clases configuradas: 16
  chuita                    → 26x.pt                         conf=0.48 sahi=False ✅
  chuita adulta             → 26x.pt                         conf=0.48 sahi=False ✅
  cushuri adulto            → 26x.pt                         conf=0.48 sahi=False ✅
  cushuri juvenil           → 26s.pt                         conf=0.48 sahi=False ✅
  gallinazo cabeza roja     → 26m.pt                         conf=0.48 sahi=False ✅
  gaviota peruana adulta    → 26m.pt                         conf=0.48 sahi=False ✅
  guanay adulto             → 26s.pt                         conf=0.48 sahi=False ✅
  pelicano adulto           → pelicano_adulto_x.pt           conf=0.21 sahi=True ✅
  pelicano juvenil          → 26s.pt            

In [17]:
# ============================================================
# 📦 CELDA 2: INSTALAR E IMPORTAR
# ============================================================
!pip install ultralytics sahi openpyxl roboflow -q

from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from collections import defaultdict
import pandas as pd
from pathlib import Path
import yaml, json, shutil, os, time
from PIL import Image
import torch

print('✅ Librerías instaladas')

✅ Librerías instaladas


In [18]:
# ============================================================
# 3. LISTAR FOTOS
# ============================================================

EXTS = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
fotos_raw = [f for f in RUTA_FOTOS.iterdir() if f.suffix in EXTS]

# 🔴 CORREGIDO: verificar que el archivo existe realmente antes de incluirlo
fotos = sorted([f for f in fotos_raw if f.exists()])
faltantes = len(fotos_raw) - len(fotos)
if faltantes > 0:
    print(f'⚠️  {faltantes} archivo(s) listado(s) pero no encontrado(s) físicamente (Drive no sincronizado)')

print(f'Total fotos válidas: {len(fotos)}')
for f in fotos[:5]:
    print(' ', f.name)
if len(fotos) > 5:
    print(f'  ... y {len(fotos)-5} más')


Total fotos válidas: 115
  371.jpg
  378.jpg
  380.jpg
  384.jpg
  389.jpg
  ... y 110 más


In [19]:
# ============================================================
# 🤖 CELDA 4: FASE 1 — PREDICCIONES PRELIMINARES
# ============================================================

# --- 4.1 Agrupar clases por modelo ---
modelo_a_clases = defaultdict(list)
for clase, cfg in CONFIG.items():
    modelo_a_clases[cfg['modelo']].append(clase)

print(f'\n📦 Modelos únicos a cargar: {len(modelo_a_clases)}')
for m, clases in modelo_a_clases.items():
    print(f'  {m} → {clases}')

# --- 4.2 Funciones de inferencia ---

def predecir_directo(ruta_foto, modelo_yolo, conf, img_w, img_h):
    """Inferencia directa con YOLOv26. Retorna lista de dicts {clase, bbox}."""
    resultados = []
    try:
        res = modelo_yolo(str(ruta_foto), conf=conf, verbose=False)[0]
        for box in res.boxes:
            cls_idx = int(box.cls.item())
            cls_name = modelo_yolo.model.names[cls_idx]
            xyxy = box.xyxy[0].cpu().numpy()  # x1, y1, x2, y2 en píxeles
            resultados.append({
                'clase': cls_name,
                'bbox': xyxy,
                'img_w': img_w,
                'img_h': img_h,
            })
    except Exception as e:
        print(f'   ⚠️ Error directo en {ruta_foto.name}: {e}')
    return resultados


def predecir_sahi(ruta_foto, ruta_modelo, conf, img_w, img_h, slice_h, slice_w, overlap):
    """Inferencia con SAHI. Retorna lista de dicts {clase, bbox}."""
    resultados = []
    try:
        detection_model = AutoDetectionModel.from_pretrained(
            model_type='ultralytics',
            model_path=str(ruta_modelo),
            confidence_threshold=conf,
            device='cuda:0',
        )
        result = get_sliced_prediction(
            str(ruta_foto),
            detection_model,
            slice_height=slice_h,
            slice_width=slice_w,
            overlap_height_ratio=overlap,
            overlap_width_ratio=overlap,
        )
        for pred in result.object_prediction_list:
            cls_name = pred.category.name
            bbox = np.array([
                pred.bbox.minx, pred.bbox.miny,
                pred.bbox.maxx, pred.bbox.maxy
            ])
            resultados.append({
                'clase': cls_name,
                'bbox': bbox,
                'img_w': img_w,
                'img_h': img_h,
            })
        del detection_model
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'   ⚠️ Error SAHI en {ruta_foto.name}: {e}')
    return resultados


def bbox_a_yolo(bbox, img_w, img_h):
    """Convierte [x1, y1, x2, y2] en formato YOLO normalizado [xc, yc, w, h]."""
    x1, y1, x2, y2 = bbox
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(img_w, x2), min(img_h, y2)
    w = x2 - x1
    h = y2 - y1
    xc = x1 + w / 2
    yc = y1 + h / 2
    return [
        xc / img_w,
        yc / img_h,
        w / img_w,
        h / img_h,
    ]


# --- 4.3 Loop principal de predicciones ---

conteos = []
detecciones_por_foto = defaultdict(list)

for modelo_nombre, clases_asignadas in modelo_a_clases.items():
    ruta_modelo = RUTA_MODELOS / modelo_nombre
    if not ruta_modelo.exists():
        print(f'\n❌ No existe modelo: {ruta_modelo}')
        continue

    # Separar clases: las que usan SAHI vs las que no
    clases_directo = [c for c in clases_asignadas if not CONFIG[c]['sahi']]
    clases_sahi    = [c for c in clases_asignadas if     CONFIG[c]['sahi']]

    print(f'\n🔄 Procesando modelo: {modelo_nombre}')
    if clases_directo:
        print(f'   Directo: {clases_directo}')
    if clases_sahi:
        print(f'   SAHI   : {clases_sahi}')

    # Cargar modelo YOLO directo SIEMPRE que haya clases sin SAHI
    modelo_yolo = None
    if clases_directo:
        print(f'   Cargando YOLO directo...')
        modelo_yolo = YOLO(str(ruta_modelo))

    for ruta_foto in fotos:
        foto_stem = ruta_foto.stem
        with Image.open(ruta_foto) as img:
            img_w, img_h = img.size

        # --- 4.3.1 Clases SIN SAHI (directo) ---
        for clase in clases_directo:
            cfg = CONFIG[clase]
            conf = cfg['conf']
            dets = predecir_directo(ruta_foto, modelo_yolo, conf, img_w, img_h)
            dets_clase = [d for d in dets if d['clase'] == clase]
            n_dets = len(dets_clase)

            conteos.append({'FOTO': foto_stem, 'Clase': clase, 'Prediccion': n_dets})

            for d in dets_clase:
                yolo_line = bbox_a_yolo(d['bbox'], img_w, img_h)
                cls_idx = CLASE_A_IDX[clase]
                detecciones_por_foto[foto_stem].append([cls_idx] + yolo_line)

        # --- 4.3.2 Clases CON SAHI ---
        for clase in clases_sahi:
            cfg = CONFIG[clase]
            conf = cfg['conf']
            slice_h = cfg.get('sahi_slice_h', SAHI_DEFAULT['slice_h'])
            slice_w = cfg.get('sahi_slice_w', SAHI_DEFAULT['slice_w'])
            overlap = cfg.get('sahi_overlap', SAHI_DEFAULT['overlap'])

            dets = predecir_sahi(ruta_foto, ruta_modelo, conf, img_w, img_h,
                                 slice_h, slice_w, overlap)
            dets_clase = [d for d in dets if d['clase'] == clase]
            n_dets = len(dets_clase)

            conteos.append({'FOTO': foto_stem, 'Clase': clase, 'Prediccion': n_dets})

            for d in dets_clase:
                yolo_line = bbox_a_yolo(d['bbox'], img_w, img_h)
                cls_idx = CLASE_A_IDX[clase]
                detecciones_por_foto[foto_stem].append([cls_idx] + yolo_line)

    # Liberar modelo YOLO de GPU
    if modelo_yolo is not None:
        del modelo_yolo
        torch.cuda.empty_cache()

print(f'\n✅ Predicciones completadas. Total filas conteo: {len(conteos)}')


# --- 4.4 Exportar Excel Preliminar ---
df_preliminar = pd.DataFrame(conteos)
df_preliminar = df_preliminar.set_index(['FOTO', 'Clase']).unstack(fill_value=0).stack().reset_index()
df_preliminar.columns = ['FOTO', 'Clase', 'Prediccion']
orden_clase = {c: i for i, c in enumerate(CLASES)}
df_preliminar['orden'] = df_preliminar['Clase'].map(orden_clase)
df_preliminar = df_preliminar.sort_values(['FOTO', 'orden']).drop('orden', axis=1).reset_index(drop=True)

# 🔥 Filtrar ceros
df_preliminar = df_preliminar[df_preliminar['Prediccion'] != 0].reset_index(drop=True)

df_preliminar.to_excel(RUTA_SALIDA, index=False)
print(f'💾 Excel preliminar guardado en: {RUTA_SALIDA}')
print(df_preliminar.head(20).to_string())


# --- 4.5 Guardar anotaciones YOLO para Roboflow ---
RUTA_TEMP_YOLO = Path('/content/temp_yolo_para_roboflow')
RUTA_TEMP_IMAGES = RUTA_TEMP_YOLO / 'images'
RUTA_TEMP_LABELS = RUTA_TEMP_YOLO / 'labels'
RUTA_TEMP_IMAGES.mkdir(parents=True, exist_ok=True)
RUTA_TEMP_LABELS.mkdir(parents=True, exist_ok=True)

for ruta_foto in fotos:
    foto_stem = ruta_foto.stem
    shutil.copy2(ruta_foto, RUTA_TEMP_IMAGES / ruta_foto.name)
    dets = detecciones_por_foto.get(foto_stem, [])
    txt_path = RUTA_TEMP_LABELS / (foto_stem + '.txt')
    with open(txt_path, 'w') as f:
        for det in dets:
            f.write(f'{det[0]} {det[1]:.6f} {det[2]:.6f} {det[3]:.6f} {det[4]:.6f}\n')

labelmap_path = RUTA_TEMP_YOLO / 'labelmap.txt'
with open(labelmap_path, 'w') as f:
    for c in CLASES:
        f.write(c + '\n')

with open(RUTA_TEMP_YOLO / 'data.yaml', 'w') as f:
    yaml.dump({'names': CLASES, 'nc': len(CLASES)}, f, allow_unicode=True)

print(f'✅ Anotaciones YOLO guardadas en: {RUTA_TEMP_YOLO}')
print(f'   Imágenes: {len(list(RUTA_TEMP_IMAGES.iterdir()))}')
print(f'   Labels  : {len(list(RUTA_TEMP_LABELS.iterdir()))}')



📦 Modelos únicos a cargar: 5
  26x.pt → ['chuita', 'chuita adulta', 'cushuri adulto', 'pichon piquero', 'pinguino adulto', 'pinguino juvenil', 'piquero juvenil', 'zarcillo']
  26s.pt → ['cushuri juvenil', 'guanay adulto', 'pelicano juvenil', 'pichon pinguino']
  26m.pt → ['gallinazo cabeza roja', 'gaviota peruana adulta']
  pelicano_adulto_x.pt → ['pelicano adulto']
  piquero_adulto_l.pt → ['piquero adulto']

🔄 Procesando modelo: 26x.pt
   Directo: ['chuita', 'chuita adulta', 'cushuri adulto', 'pichon piquero', 'pinguino adulto', 'pinguino juvenil', 'piquero juvenil']
   SAHI   : ['zarcillo']
   Cargando YOLO directo...
Performing prediction on 16 slices.
Performing prediction on 12 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing 

The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.


✅ Anotaciones YOLO guardadas en: /content/temp_yolo_para_roboflow
   Imágenes: 115
   Labels  : 115


In [21]:
# -*- coding: utf-8 -*-
"""
🔍 Verificación de conexión a Roboflow
Ejecuta ESTA celda antes de la Fase 2 para confirmar que todo está OK.
"""

# ============================================================
# 2. INSTALAR (si aún no lo hiciste)
# ============================================================
!pip install roboflow -q

from roboflow import Roboflow
import sys

print("=" * 60)
print("🔍 VERIFICACIÓN DE CONEXIÓN A ROBOFLOW")
print("=" * 60)

# ============================================================
# 3. PROBAR AUTENTICACIÓN
# ============================================================
print("\n[1/4] Probando autenticación con API Key...")
try:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    print("   ✅ API Key válida. Usuario autenticado.")
except Exception as e:
    print(f"   ❌ ERROR de autenticación: {e}")
    print("   💡 Causas comunes:")
    print("      - La API Key está mal copiada (faltan/le sobran caracteres)")
    print("      - La API Key fue revocada o regenerada")
    print("      - No tienes conexión a internet en Colab")
    sys.exit(1)

# ============================================================
# 4. PROBAR ACCESO AL WORKSPACE
# ============================================================
print(f"\n[2/4] Buscando workspace: '{ROBOFLOW_WORKSPACE}'...")
try:
    workspace = rf.workspace(ROBOFLOW_WORKSPACE)
    print(f"   ✅ Workspace encontrado.")

    # Listar proyectos del workspace para confirmar acceso
    proyectos = workspace.projects()
    nombres_proyectos = [p['name'] if isinstance(p, dict) else str(p) for p in proyectos]
    print(f"   📋 Proyectos visibles en este workspace ({len(nombres_proyectos)}):")
    for np in nombres_proyectos[:10]:
        print(f"      • {np}")
    if len(nombres_proyectos) > 10:
        print(f"      ... y {len(nombres_proyectos)-10} más")

except Exception as e:
    print(f"   ❌ ERROR accediendo al workspace: {e}")
    print("   💡 Causas comunes:")
    print("      - El nombre del workspace está mal escrito (es case-sensitive)")
    print("      - Tu API Key no tiene permisos sobre este workspace")
    print("      - El workspace no existe")
    sys.exit(1)

# ============================================================
# 5. PROBAR ACCESO AL PROYECTO
# ============================================================
print(f"\n[3/4] Buscando proyecto: '{ROBOFLOW_PROJECT}'...")
try:
    project = workspace.project(ROBOFLOW_PROJECT)
    print(f"   ✅ Proyecto encontrado: {project.name}")
    print(f"   📊 Tipo: {project.type}")
    print(f"   📊 Clases: {project.classes}")
    print(f"   📊 Versiones existentes: {project.versions()}")
except Exception as e:
    print(f"   ❌ ERROR accediendo al proyecto: {e}")
    print("   💡 Causas comunes:")
    print("      - El nombre del proyecto está mal escrito (es case-sensitive)")
    print("      - El proyecto no existe en este workspace")
    print("      - No tienes permisos de escritura en el proyecto")
    sys.exit(1)

# ============================================================
# 6. PROBAR SUBIDA DE UNA IMAGEN DE PRUEBA
# ============================================================
print(f"\n[4/4] Probando subida de 1 imagen de prueba...")
print("   (Si tienes fotos en tu carpeta, usaremos la primera)")

from pathlib import Path
RUTA_FOTOS = Path('/content/drive/MyDrive/conteo de aves marinas/fotos')

if not RUTA_FOTOS.exists():
    print(f"   ⚠️ No encontré la carpeta de fotos: {RUTA_FOTOS}")
    print("   Saltando prueba de subida. La conexión básica ya está OK.")
else:
    fotos = sorted([f for f in RUTA_FOTOS.iterdir()
                    if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])
    if not fotos:
        print("   ⚠️ No hay fotos en la carpeta. Saltando prueba de subida.")
    else:
        foto_prueba = fotos[0]
        print(f"   📷 Usando imagen de prueba: {foto_prueba.name}")
        try:
            project.upload(
                image_path=str(foto_prueba),
                batch_name='test_conexion',
                num_retry_uploads=1,
            )
            print("   ✅ Imagen de prueba subida correctamente.")
            print("   🗑️  Puedes borrarla manualmente de Roboflow si no la necesitas.")
        except Exception as e:
            print(f"   ❌ ERROR subiendo imagen: {e}")
            print("   💡 La conexión básica funciona, pero la subida falló.")
            print("      Revisa que el proyecto tenga espacio y permisos de escritura.")

# ============================================================
# 7. RESUMEN FINAL
# ============================================================
print("\n" + "=" * 60)
print("📋 RESUMEN DE VERIFICACIÓN")
print("=" * 60)
print(f"   API Key      : {'✅ Válida' if 'rf' in locals() else '❌ Falló'}")
print(f"   Workspace    : {ROBOFLOW_WORKSPACE} {'✅' if 'workspace' in locals() else '❌'}")
print(f"   Proyecto     : {ROBOFLOW_PROJECT} {'✅' if 'project' in locals() else '❌'}")
print(f"   Subida test  : {'✅ OK (o no probada)' if 'foto_prueba' not in locals() else '✅ OK'}")
print("\n🚀 Si todo sale ✅, ya puedes ejecutar la Celda 5 (FASE 2) con confianza.")

🔍 VERIFICACIÓN DE CONEXIÓN A ROBOFLOW

[1/4] Probando autenticación con API Key...
   ✅ API Key válida. Usuario autenticado.

[2/4] Buscando workspace: 'jhon-goicochea'...
loading Roboflow workspace...
   ✅ Workspace encontrado.
   📋 Proyectos visibles en este workspace (1):
      • jhon-goicochea/conteo-av-1

[3/4] Buscando proyecto: 'conteo-av-1'...
loading Roboflow project...
   ✅ Proyecto encontrado: conteo-av-1
   📊 Tipo: object-detection
   📊 Clases: {}
   📊 Versiones existentes: []

[4/4] Probando subida de 1 imagen de prueba...
   (Si tienes fotos en tu carpeta, usaremos la primera)
   📷 Usando imagen de prueba: 371.jpg
   ✅ Imagen de prueba subida correctamente.
   🗑️  Puedes borrarla manualmente de Roboflow si no la necesitas.

📋 RESUMEN DE VERIFICACIÓN
   API Key      : ✅ Válida
   Workspace    : jhon-goicochea ✅
   Proyecto     : conteo-av-1 ✅
   Subida test  : ✅ OK

🚀 Si todo sale ✅, ya puedes ejecutar la Celda 5 (FASE 2) con confianza.


In [22]:
# ============================================================
# ☁️ CELDA 5: FASE 2 — SUBIR A ROBOFLOW + GUARDAR MAPEO DE NOMBRES
# ============================================================
from roboflow import Roboflow
import json

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace(ROBOFLOW_WORKSPACE)
project = workspace.project(ROBOFLOW_PROJECT)

print(f'Conectado a Roboflow: {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}')

subidas_ok = 0
subidas_fail = 0

# Mapeo: nombre_base_roboflow (sin hash) -> nombre_original (stem)
# Roboflow convierte 110.jpg -> 110_jpg.rf.XXXX.jpg
# Guardamos: "110_jpg" -> "110"
mapeo_nombres = {}

for ruta_foto in fotos:
    foto_stem = ruta_foto.stem
    img_dest = RUTA_TEMP_IMAGES / ruta_foto.name
    lbl_dest = RUTA_TEMP_LABELS / (foto_stem + '.txt')

    # Solo subir anotaciones si el archivo existe y NO está vacío
    annotation_path = None
    labelmap_path_str = None

    if lbl_dest.exists() and lbl_dest.stat().st_size > 0:
        annotation_path = str(lbl_dest)
        labelmap_path_str = str(labelmap_path)

    try:
        project.upload(
            image_path=str(img_dest),
            annotation_path=annotation_path,           # None si está vacío
            annotation_labelmap=labelmap_path_str,     # None si está vacío
            batch_name=ROBOFLOW_BATCH_NAME,
            is_prediction=True,
            num_retry_uploads=2,
        )
        subidas_ok += 1

        # Guardar mapeo para recuperar nombre original después
        ext_limpia = ruta_foto.suffix.lower().lstrip('.')
        stem_roboflow_esperado = f"{foto_stem}_{ext_limpia}"
        mapeo_nombres[stem_roboflow_esperado] = foto_stem

    except Exception as e:
        subidas_fail += 1
        if subidas_fail <= 3:
            print(f'   ⚠️ Error subiendo {foto_stem}: {e}')

    if subidas_ok % 10 == 0:
        print(f'   ... {subidas_ok}/{len(fotos)} imágenes subidas')

# Guardar mapeo en Drive para usarlo en la Celda 6
RUTA_MAPEO = RUTA_FOTOS.parent / 'mapeo_roboflow_nombres.json'
with open(RUTA_MAPEO, 'w', encoding='utf-8') as f:
    json.dump(mapeo_nombres, f, indent=2, ensure_ascii=False)

print(f'\n✅ Subidas OK: {subidas_ok} | Fallos: {subidas_fail}')
print(f'💾 Mapeo de nombres guardado en: {RUTA_MAPEO}')
print(f'📋 Ve a Roboflow → Annotate → Revisa el batch: "{ROBOFLOW_BATCH_NAME}"')
print('   ⚠️ IMPORTANTE: al corregir/anotar, asegúrate de seleccionar la CLASE CORRECTA')
print('      en el dropdown antes de dibujar cada caja. La clase por defecto es "chuita".')

loading Roboflow workspace...
loading Roboflow project...
Conectado a Roboflow: jhon-goicochea/conteo-av-1
   ... 10/115 imágenes subidas
   ... 20/115 imágenes subidas
   ... 30/115 imágenes subidas
   ... 40/115 imágenes subidas
   ... 50/115 imágenes subidas
   ... 60/115 imágenes subidas
   ... 70/115 imágenes subidas
   ... 80/115 imágenes subidas
   ... 90/115 imágenes subidas
   ... 100/115 imágenes subidas
   ... 110/115 imágenes subidas

✅ Subidas OK: 115 | Fallos: 0
💾 Mapeo de nombres guardado en: /content/drive/MyDrive/conteo de aves marinas/mapeo_roboflow_nombres.json
📋 Ve a Roboflow → Annotate → Revisa el batch: "conteo_preliminar"
   ⚠️ IMPORTANTE: al corregir/anotar, asegúrate de seleccionar la CLASE CORRECTA
      en el dropdown antes de dibujar cada caja. La clase por defecto es "chuita".


In [23]:
# ============================================================
# 📥 CELDA 6: FASE 3 — DESCARGAR VALIDADOS Y RECONTAR
# ============================================================
# EJECUTA ESTA CELDA DESPUÉS DE HABER CORREGIDO EN ROBOFLOW
# Y HABER GENERADO UNA NUEVA VERSIÓN DEL DATASET

ROBOFLOW_VERSION = 1   # ← Cambia al número de versión que generaste en Roboflow

print(f'\n📥 Descargando versión {ROBOFLOW_VERSION} desde Roboflow...')

from roboflow import Roboflow
import yaml
from pathlib import Path
from collections import defaultdict
import pandas as pd
import json

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)

dataset_obj = version.download("yolov8")
dataset_path = dataset_obj.location
print(f'✅ Dataset descargado en: {dataset_path}')

# --- 1. Cargar mapeo de nombres ---
RUTA_MAPEO = RUTA_FOTOS.parent / 'mapeo_roboflow_nombres.json'
if RUTA_MAPEO.exists():
    with open(RUTA_MAPEO, 'r', encoding='utf-8') as f:
        mapeo_nombres = json.load(f)
    print(f'📋 Mapeo de nombres cargado: {len(mapeo_nombres)} entradas')
else:
    # Fallback: reconstruir mapeo desde los nombres originales
    print('⚠️ No se encontró mapeo guardado. Reconstruyendo desde fotos originales...')
    mapeo_nombres = {}
    for ruta_foto in fotos:
        ext_limpia = ruta_foto.suffix.lower().lstrip('.')
        stem_roboflow_esperado = f"{ruta_foto.stem}_{ext_limpia}"
        mapeo_nombres[stem_roboflow_esperado] = ruta_foto.stem

# --- 2. Leer data.yaml descargado (orden REAL de clases en Roboflow) ---
data_yaml_path = Path(dataset_path) / 'data.yaml'
with open(data_yaml_path, 'r', encoding='utf-8') as f:
    data_yaml = yaml.safe_load(f)

names_from_yaml = data_yaml.get('names', {})
if isinstance(names_from_yaml, dict):
    names_list = [names_from_yaml[i] for i in sorted(names_from_yaml, key=int)]
else:
    names_list = list(names_from_yaml)

print(f'📋 Clases en data.yaml descargado ({len(names_list)}):')
for i, n in enumerate(names_list):
    print(f'   [{i}] {n}')

# Verificar coincidencia con CLASES
clases_faltantes = set(CLASES) - set(names_list)
clases_extra = set(names_list) - set(CLASES)
if clases_faltantes:
    print(f'⚠️ Clases en tu config pero NO en Roboflow: {clases_faltantes}')
if clases_extra:
    print(f'⚠️ Clases en Roboflow pero NO en tu config: {clases_extra}')

# Mapeo índice -> nombre de clase (usando el orden EXACTO del data.yaml descargado)
idx_to_clase = {i: names_list[i] for i in range(len(names_list))}

# --- 3. Recontar desde anotaciones descargadas ---
conteos_validados = []

splits = ['train', 'valid', 'test']
for split in splits:
    labels_dir = Path(dataset_path) / split / 'labels'
    if not labels_dir.exists():
        print(f'   ⚠️ No existe carpeta: {labels_dir}')
        continue

    txt_files = sorted(labels_dir.glob('*.txt'))
    print(f'   📁 {split}: {len(txt_files)} archivos .txt')

    for txt_file in txt_files:
        # Nombre hasheado: 110_jpg.rf.273b53805f7071c41578479657232342.txt
        foto_stem_hash = txt_file.stem
        stem_base = foto_stem_hash.split('.rf.')[0]  # 110_jpg
        foto_nombre = mapeo_nombres.get(stem_base, stem_base)  # 110

        conteo_foto = defaultdict(int)
        with open(txt_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                cls_idx = int(parts[0])
                clase_detectada = idx_to_clase.get(cls_idx, f'clase_desconocida_{cls_idx}')
                conteo_foto[clase_detectada] += 1

        # Asegurar que todas las clases de CLASES aparezcan (con 0 si no hay)
        for clase in CLASES:
            conteos_validados.append({
                'FOTO': foto_nombre,
                'Clase': clase,
                'Prediccion': conteo_foto.get(clase, 0),
            })

# --- 4. Ordenar y exportar ---
df_validado = pd.DataFrame(conteos_validados)
orden_clase = {c: i for i, c in enumerate(CLASES)}
df_validado['orden'] = df_validado['Clase'].map(orden_clase)
df_validado = df_validado.sort_values(['FOTO', 'orden']).drop('orden', axis=1).reset_index(drop=True)

# 🔥 Filtrar ceros
df_validado = df_validado[df_validado['Prediccion'] != 0].reset_index(drop=True)

df_validado.to_excel(RUTA_VALIDADO, index=False)
print(f'\n💾 Excel validado guardado en: {RUTA_VALIDADO}')
print(f'   Total filas: {len(df_validado)}')
print(df_validado.head(20).to_string())

# Limpiar temp si se desea (descomenta si quieres)
# import shutil
# shutil.rmtree(RUTA_TEMP_YOLO)
# shutil.rmtree(dataset_path)


📥 Descargando versión 1 desde Roboflow...
loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 95.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to conteo-av-1-1 in yolov8:: 100%|██████████| 235/235 [00:00<00:00, 652.19it/s]


✅ Dataset descargado en: /content/conteo-av-1-1
📋 Mapeo de nombres cargado: 115 entradas
📋 Clases en data.yaml descargado (8):
   [0] cushuri adulto
   [1] pelicano adulto
   [2] pelicano juvenil
   [3] pinguino adulto
   [4] pinguino juvenil
   [5] piquero adulto
   [6] piquero juvenil
   [7] zarcillo
⚠️ Clases en tu config pero NO en Roboflow: {'guanay adulto', 'pichon piquero', 'gallinazo cabeza roja', 'gaviota peruana adulta', 'pichon pinguino', 'chuita', 'cushuri juvenil', 'chuita adulta'}
   📁 train: 81 archivos .txt
   📁 valid: 23 archivos .txt
   📁 test: 11 archivos .txt

💾 Excel validado guardado en: /content/drive/MyDrive/conteo de aves marinas/Conteos_validados.xlsx
   Total filas: 335
   FOTO            Clase  Prediccion
0   371  pinguino adulto           3
1   371   piquero adulto           2
2   371         zarcillo          38
3   378  pelicano adulto           1
4   378   piquero adulto           1
5   378         zarcillo          54
6   380  pinguino adulto           